### Conversation Q&A Chatbot
In many Q&A applications we want to allow the user to have a back-and-forth conversation, meaning the application needs some sort of "memory" of past questions and answers, and some logic for incorporating those into its current thinking.

In this guide we focus on adding logic for incorporating historical messages. Further details on chat history management is covered in the previous videos.

We will cover two approaches:

- Chains, in which we always execute a retrieval step;
- Agents, in which we give an LLM discretion over whether and how to execute a retrieval step (or multiple steps).

> **Note (LangChain 1.x update):** The original notebook used `langchain.chains.create_retrieval_chain`, `langchain.chains.combine_documents.create_stuff_documents_chain`, and `langchain.chains.create_history_aware_retriever`. These helper functions were removed from `langchain.chains` in LangChain 1.x (the `langchain.chains` module no longer contains them). This version rebuilds the exact same RAG pipelines using LangChain Expression Language (LCEL) primitives from `langchain_core` (`RunnablePassthrough`, `RunnableLambda`, `StrOutputParser`), which remain fully supported in LangChain 1.3.9. The overall behaviour, inputs, and outputs are unchanged.

In [34]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq

groq_api_key=os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant")

llm


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000257FE6F2320>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000257FE6F2650>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [35]:
#!pip install bs4

In [36]:
os.environ['HF_TOKEN']=os.getenv("HF_TOKEN")
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 703.52it/s]


In [37]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_text_splitters import RecursiveCharacterTextSplitter

# LCEL building blocks used to replace the old langchain.chains helpers
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [38]:
# 1. Load, chunk and index the contents of the blog to create a retriever.
import bs4
loader = WebBaseLoader(
    web_paths=("https://www.healthline.com/nutrition/plant-based-diet-guide",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

docs=loader.load()
docs

[Document(metadata={'source': 'https://www.healthline.com/nutrition/plant-based-diet-guide'}, page_content='')]

In [ ]:
# Split documents and verify embeddings before upsert to Chroma
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
print("splits count:", len(splits))
if not splits:
    raise RuntimeError("No splits produced from documents. Inspect `docs` content.")
# Preview first split to ensure content exists
print("first split preview:", splits[0].page_content[:400])
# Verify embeddings backend is available and returns non-empty vectors
import os
print("HF_TOKEN present:", bool(os.getenv("HF_TOKEN")))
try:
    sample_texts = [splits[0].page_content]
    sample_emb = embeddings.embed_documents(sample_texts)
    print("sample_emb type:", type(sample_emb), "len:", len(sample_emb))
except Exception as e:
    raise RuntimeError("Embedding call failed: " + str(e))
if not sample_emb or (isinstance(sample_emb, list) and len(sample_emb) == 0):
    raise RuntimeError("Embeddings returned empty. Check HF_TOKEN, model availability, and network.")
# Safe to create vectorstore now
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever()
retriever

ValueError: Expected Embeddings to be non-empty list or numpy array, got [] in upsert.

In [ ]:
## Prompt Template
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

# Helper used to "stuff" retrieved documents into the {context} placeholder,
# equivalent to what create_stuff_documents_chain did internally.
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
# Replaces: question_answer_chain = create_stuff_documents_chain(llm, prompt) 
question_answer_chain = (
    RunnablePassthrough.assign(context=lambda x: format_docs(x["context"]))
    | prompt
    | llm
    | StrOutputParser()
)

# Replaces: rag_chain = create_retrieval_chain(retriever, question_answer_chain)
rag_chain = RunnablePassthrough.assign(
    context=(lambda x: x["input"]) | retriever
).assign(answer=question_answer_chain)

In [ ]:
response=rag_chain.invoke({"input":"What is Plant Based diet?"})
response

{'input': 'What is Self-Reflection',
 'context': [Document(id='0c56307e-c71c-45dc-9cee-4f4f4df9c682', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Self-Reflection#\nSelf-reflection is a vital aspect that allows autonomous agents to improve iteratively by refining past action decisions and correcting previous mistakes. It plays a crucial role in real-world tasks where trial and error are inevitable.\nReAct (Yao et al. 2023) integrates reasoning and acting within LLM by extending the action space to be a combination of task-specific discrete actions and the language space. The former enables LLM to interact with the environment (e.g. use Wikipedia search API), while the latter prompting LLM to generate reasoning traces in natural language.\nThe ReAct prompt template incorporates explicit steps for LLM to think, roughly formatted as:\nThought: ...\nAction: ...\nObservation: ...\n... (Repeated many times)'),
  Document(id='1cccc104-a804-40a6-94

In [ ]:
response['answer']

'Self-reflection is a vital aspect that allows autonomous agents to improve iteratively by refining past action decisions and correcting previous mistakes. It plays a crucial role in real-world tasks where trial and error are inevitable. This process enables agents to learn from their failures and adapt to new situations.'

In [ ]:
rag_chain.invoke({"input":"What are the benefits of plant based diet?"})

{'input': 'Howw do we achieve it',
 'context': [Document(id='381c16ad-0f15-489b-a495-b1f7502bf42a', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='}\n]\nChallenges#\nAfter going through key ideas and demos of building LLM-centered agents, I start to see a couple common limitations:'),
  Document(id='8ad63458-b7fa-4e25-91bb-bd29b7537c6a', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='}\n]\nChallenges#\nAfter going through key ideas and demos of building LLM-centered agents, I start to see a couple common limitations:'),
  Document(id='6024e4d4-50f8-477c-a755-94311fb41092', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Resources:\n1. Internet access for searches and information gathering.\n2. Long Term memory management.\n3. GPT-3.5 powered Agents for delegation of simple tasks.\n4. File output.\n\nPerformance Evaluation:\n1. Continuously review and analyz

### Adding Chat History

In [ ]:
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)
contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

In [ ]:
# Replaces: history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_q_prompt)
contextualize_q_chain = contextualize_q_prompt | llm | StrOutputParser()

def contextualized_question(input: dict):
    # If there is chat history, rewrite the question to be a standalone
    # question using the contextualize prompt; otherwise pass it through.
    if input.get("chat_history"):
        return contextualize_q_chain
    else:
        return input["input"]

history_aware_retriever = RunnableLambda(contextualized_question) | retriever
history_aware_retriever

RunnableLambda(contextualized_question)
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x00000257FE613910>, search_kwargs={})

In [ ]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

In [ ]:
# Replaces: question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
question_answer_chain = (
    RunnablePassthrough.assign(context=lambda x: format_docs(x["context"]))
    | qa_prompt
    | llm
    | StrOutputParser()
)

# Replaces: rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)
rag_chain = RunnablePassthrough.assign(
    context=history_aware_retriever
).assign(answer=question_answer_chain)

In [ ]:
from langchain_core.messages import AIMessage,HumanMessage
chat_history=[]
question="What is Plant Based diet?"
response1=rag_chain.invoke({"input":question,"chat_history":chat_history})

chat_history.extend(
    [
        HumanMessage(content=question),
        AIMessage(content=response1["answer"])
    ]
)

question2="Tell me more about the choices for plant based diet?"
response2=rag_chain.invoke({"input":question,"chat_history":chat_history})
print(response2['answer'])

Self-Reflection is a process that allows autonomous agents to reflect on their past actions, identify mistakes, and correct them to make better decisions in the future.


In [ ]:
chat_history

[HumanMessage(content='What is Self-Reflection', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Self-Reflection is a vital aspect that allows autonomous agents to improve iteratively by refining past action decisions and correcting previous mistakes. It is crucial in real-world tasks where trial and error are inevitable. Self-Reflection helps agents learn from their mistakes and make better decisions in the future.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [ ]:
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

C:\Users\kushp\AppData\Roaming\Python\Python310\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [ ]:
conversational_rag_chain.invoke(
    {"input": "What is the meal plan for a plant based diet?"},
    config={
        "configurable": {"session_id": "abc123"}
    },  # constructs a key "abc123" in `store`.
)["answer"]

'Task decomposition is the process of breaking down a complex task into smaller, manageable subtasks or steps. This helps to organize and plan the task, making it easier to achieve the desired outcome. It involves identifying the key components of the task and determining the necessary actions required to complete it.'

In [ ]:
conversational_rag_chain.invoke(
    {"input": "How can this improve in losing extra weight?"},
    config={"configurable": {"session_id": "abc123"}},
)["answer"]

'Common ways of task decomposition include: \n\n1. Using simple prompting with a language model (LLM) like "Steps for XYZ.\\n1." or "What are the subgoals for achieving XYZ?"\n2. Utilizing task-specific instructions, such as "Write a story outline." for writing a novel.\n3. Using human inputs to guide the decomposition process.\n4. Employing external classical planners, as seen in the LLM+P approach.'

In [ ]:
store

{'abc123': InMemoryChatMessageHistory(messages=[HumanMessage(content='What is Task Decomposition?', additional_kwargs={}, response_metadata={}), AIMessage(content='Task decomposition is the process of breaking down a complex task into smaller, manageable subtasks or steps. This helps to organize and plan the task, making it easier to achieve the desired outcome. It involves identifying the key components of the task and determining the necessary actions required to complete it.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What are common ways of doing it?', additional_kwargs={}, response_metadata={}), AIMessage(content='Common ways of task decomposition include: \n\n1. Using simple prompting with a language model (LLM) like "Steps for XYZ.\\n1." or "What are the subgoals for achieving XYZ?"\n2. Utilizing task-specific instructions, such as "Write a story outline." for writing a novel.\n3. Using human inputs to guide the deco

In [41]:
docs = loader.load()
print("docs count:", len(docs))
if len(docs)>0:
    print(docs[0].page_content[:400])
else:
    print("No documents loaded — parse selectors likely didn't match the page.")

docs count: 1



In [42]:
# Replace your loader definition with this (remove bs_kwargs parse_only)
loader = WebBaseLoader(
    web_paths=("https://www.healthline.com/nutrition/plant-based-diet-guide",)
)
docs = loader.load()
print("docs count after relaxed loader:", len(docs))

docs count after relaxed loader: 1


In [43]:
if not docs:
    raise RuntimeError("Loader returned no documents. Try removing/adjusting bs_kwargs or fetch page manually (requests + BeautifulSoup) to inspect structure.")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
print("splits count:", len(splits))
if not splits:
    raise RuntimeError("No splits produced from documents. Inspect `docs` content.")

splits count: 25


In [ ]:
# ensure HF token available if using HuggingFaceEmbeddings
print("HF_TOKEN present:", bool(os.getenv("HF_TOKEN")))

# compute embeddings for a sample to verify embedding works
sample_texts = [splits[0].page_content[:100]]
sample_emb = embeddings.embed_documents(sample_texts)
print("sample_emb length:", len(sample_emb), "shape example:", type(sample_emb[0]))

# if ok, create vectorstore
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever()
print("Vectorstore created, retriever ready.")

HF_TOKEN present: True
sample_emb length: 1 shape example: <class 'list'>
